Load and split dataframe

In [1]:
import pandas as pd
import sys
sys.path.append("..")
from python_editor.data_processing import split_by_developer, get_vectorized_features_and_label

df = pd.read_pickle("../data/features_v2.pkl")
df = df.drop(columns=["embedding"])
df.to_pickle("../data/features_v3.pkl")

train, test = split_by_developer(df, test_size=0.3, random_state=0)

Stack all features to get a vector for each data point

In [2]:
from python_editor.feature_generation_v2 import TRANSFORMED_FEATURES

X_train, y_train = get_vectorized_features_and_label(train, TRANSFORMED_FEATURES)
X_test, y_test = get_vectorized_features_and_label(test, TRANSFORMED_FEATURES)

Train a random-forest-regressor on generated features only. We use optuna for hyperparameter tuning and mlflow for experiment tracking

In [3]:
import optuna
import mlflow
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from python_editor.model_evaluation import get_metrics

mlflow.set_tracking_uri("sqlite:///models/mlflow/mlflow.db")
mlflow.create_experiment("random_forest_regressor", artifact_location="models/mlflow/artifacts")
mlflow.set_experiment("random_forest_regressor")

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 3, 50),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "oob_score": True,
        "random_state": 0,
        "n_jobs": -1
    }

    model = RandomForestRegressor(**params)

    cv = KFold(n_splits=5, shuffle=True, random_state=0)

    # negative RMSE because sklearn maximizes score
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    rmse = -scores.mean()

    # Nested MLflow run for each trial
    with mlflow.start_run(nested=True):
        mlflow.log_params(params)
        mlflow.log_metric("cv_rmse", rmse)

    return rmse


with mlflow.start_run(run_name="optuna_rf_search"):
    mlflow.set_tags(
        {
            "dataset": "features_v3.pkl",
            "processing": "data_processing.py",
            "feature_generation": "feature_generation_v2.py",
            "test_size": 0.3,
            "random_state": 0
        }
    )

    study = optuna.create_study(direction="minimize")

    study.optimize(objective, n_trials=50, show_progress_bar=True)

    best_params = study.best_params

    # Final model
    best_model = RandomForestRegressor(
        **best_params,
        oob_score=True,
        random_state=0,
        n_jobs=-1
    )

    best_model.fit(X_train, y_train)

    preds = best_model.predict(X_test)
    metrics = get_metrics(y_test, preds)

    # Log best results
    mlflow.log_params(best_params)
    mlflow.log_metric("best_cv_rmse", study.best_value)
    mlflow.log_metric("test_mae", metrics["MAE"])
    mlflow.log_metric("test_rmse", metrics["RMSE"])
    mlflow.log_metric("test_r2", metrics["R2"])

    mlflow.sklearn.log_model(best_model, "model")

print("Best Params:", best_params)
print(metrics)


2026/05/17 19:33:52 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/17 19:33:52 INFO mlflow.store.db.utils: Updating database tables
[I 2026-05-17 19:33:58,192] A new study created in memory with name: no-name-dc3c0380-2f51-4850-9166-b1ae592b937f


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-05-17 19:34:03,867] Trial 0 finished with value: 2.5220023666636657 and parameters: {'n_estimators': 497, 'max_depth': 40, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_features': None}. Best is trial 0 with value: 2.5220023666636657.
[I 2026-05-17 19:34:06,325] Trial 1 finished with value: 2.5916781856103617 and parameters: {'n_estimators': 276, 'max_depth': 7, 'min_samples_split': 12, 'min_samples_leaf': 8, 'max_features': 'log2'}. Best is trial 0 with value: 2.5220023666636657.
[I 2026-05-17 19:34:08,362] Trial 2 finished with value: 2.501408132501378 and parameters: {'n_estimators': 220, 'max_depth': 14, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 2 with value: 2.501408132501378.
[I 2026-05-17 19:34:09,428] Trial 3 finished with value: 2.521075687701675 and parameters: {'n_estimators': 303, 'max_depth': 45, 'min_samples_split': 9, 'min_samples_leaf': 7, 'max_features': 'sqrt'}. Best is trial 2 with value: 2.501408132501378.
[I

2026/05/17 19:35:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/17 19:35:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Best Params: {'n_estimators': 410, 'max_depth': 44, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 'log2'}
{'MAE': '2.045', 'RMSE': '2.598', 'R2': '0.369'}


Register the model for easy access in testing and production

In [4]:
client = mlflow.MlflowClient()
experiment = client.get_experiment_by_name("random_forest_regressor")

run_name = "optuna_rf_search"
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"attributes.run_name = '{run_name}'",
    max_results=1
)

parent_run = runs[0]
run_id = parent_run.info.run_id
run = mlflow.get_run(run_id)

model_name = "recommendation_model"

# Register model
result = mlflow.register_model(
    model_uri=f"runs:/{run_id}/model",
    name=model_name
)

# Assign alias
client.set_registered_model_alias(
    name=model_name,
    alias="production",
    version=result.version
)

Successfully registered model 'recommendation_model'.
2026/05/17 19:35:30 WARNING mlflow.tracking._model_registry.fluent: Run with id 1cc5aa67126d4c99958ea79eea347da8 has no artifacts at artifact path 'model', registering model based on models:/m-00beeba9e71045a6a444d640a1b3420c instead
Created version '1' of model 'recommendation_model'.


We see that our third model is a little worse than the second one but we got rid of embbeddings burden which decreases latency and increases interpretability 

In [5]:
import pickle
from python_editor.model_evaluation import get_metrics

df_v2 = pd.read_pickle("../data/features_v2.pkl")
train_v2, test_v2 = split_by_developer(df_v2, test_size=0.3, random_state=0)
X_test_v2, y_test_v2 = get_vectorized_features_and_label(test_v2, TRANSFORMED_FEATURES)

with open("models/model_v2.pkl", "rb") as f:
    model_v2 = pickle.load(f)


y_train_model_v2 = model_v2.oob_prediction_
y_train_model_v3 = best_model.oob_prediction_

y_test_model_v2 = model_v2.predict(X_test_v2)
y_test_model_v3 = best_model.predict(X_test)

print(f"Train v2:  {get_metrics(y_train, y_train_model_v2)}\nTest  v2:  {get_metrics(y_test_v2, y_test_model_v2)}")
print(f"Train v3:  {get_metrics(y_train, y_train_model_v3)}\nTest  v3:  {get_metrics(y_test, y_test_model_v3)}")

Train v2:  {'MAE': '1.932', 'RMSE': '2.431', 'R2': '0.367'}
Test  v2:  {'MAE': '1.974', 'RMSE': '2.458', 'R2': '0.434'}
Train v3:  {'MAE': '1.929', 'RMSE': '2.459', 'R2': '0.352'}
Test  v3:  {'MAE': '2.045', 'RMSE': '2.598', 'R2': '0.369'}


We save the new model

In [6]:
with open("models/model_v3.pkl", "wb") as f:
    pickle.dump(best_model, f)

We create a function that takes text input, processes it, generate features and use the model to predict

In [7]:
from python_editor.model_prediction import get_model_prediction_from_text
from python_editor.feature_generation_v2 import generate_transformed_features

row = pd.Series({"text": "import pandas as pd\nif True:\n\tpass"})

features, score = get_model_prediction_from_text(row, generate_transformed_features, best_model, embedding_dim=0)
features, score

({'characters': np.float64(3.5553480614894135),
  'code_compactness': 1.0,
  'line_length_std': np.float64(1.9485480992350324),
  'cyclomatic_complexity': np.float64(0.6931471805599453),
  'long_line': 0,
  'bad_name': 0,
  'comment_ratio': 0.0,
  'has_docstring': 0,
  'variable_density': 0.0,
  'func_density': 0.0,
  'too_many_args': 0,
  'class_density': 0.0,
  'avg_class_methods': 0,
  'func_class_docstring_ratio': 0,
  'unused_imports': 1},
 2.2430216968620376)